In [1]:
import torch
import torch.nn as nn
import numpy as np
import math
import matplotlib.pyplot as plt

/home/ubuntu/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [3]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-02-13 11:17:11--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8000::154, 2606:50c0:8001::154, 2606:50c0:8002::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8000::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  5.62MB/s    in 0.2s    

2026-02-13 11:17:11 (5.62 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [2]:
with open("input.txt", "r", encoding="utf-8") as f:
    text= f.read()

print("Length: ", len(text))
print(text[:500])

Length:  1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [3]:
chars= sorted(list(set(text)))
vocab_size= len(chars)

print("Vocab size: ", vocab_size)
print(chars[:10])

Vocab size:  65
['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3']


In [4]:
stoi= {ch:i for i, ch in enumerate(chars)}
itos= {i:ch for i, ch in enumerate(chars)}



In [5]:
def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return "".join([itos[i] for i in l])

In [6]:
encoded= encode("Hello")
print(encoded)
print(decode(encoded))

[20, 43, 50, 50, 53]
Hello


In [7]:
data= torch.tensor(encode(text), dtype= torch.long)
print(data.shape)

torch.Size([1115394])


In [8]:
n= int(0.9* len(data))
train_data= data[:n]
test_data= data[n:]

In [9]:
device= "cuda" if torch.cuda.is_available() else "cpu"

vocab_size= len(chars)
block_size= 128
batch_size= 32

d_model= 192
n_heads= 4
n_layers= 6
dff= 4*d_model

learning_rate= 3e-4
max_iters= 5000*2
eval_intervals= 500

In [10]:

import random

def get_batch(split):
    data_source= train_data if split=='train' else test_data
    ix= torch.randint(len(data_source)-block_size, (batch_size, ))

    x= torch.stack([data_source[i:i+block_size] for i in ix])
    y= torch.stack([data_source[i+1:i+block_size+1] for i in ix])

    x = torch.clamp(x, 0, vocab_size - 1)
    y = torch.clamp(y, 0, vocab_size - 1)
    return x.to(device), y.to(device)

In [36]:
class GPTBlock(nn.Module):
    def __init__(self, d_model, n_heads, dff):
        super().__init__()

        self.norm1= nn.LayerNorm(d_model)
        self.attn= nn.MultiheadAttention(d_model, n_heads, batch_first= True, dropout= 0.1)

        self.norm2= nn.LayerNorm(d_model)
        self.ff= nn.Sequential(
            nn.Linear(d_model, dff),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(dff, d_model)
        )

    def forward(self, x):
        mask= nn.Transformer.generate_square_subsequent_mask(x.size(1)).to(x.device)
        x_norm= self.norm1(x)
        attn_out, _= self.attn(x_norm, x_norm, x_norm, attn_mask= mask)

        x= x+ attn_out

        x= x+ self.ff(self.norm2(x))

        return x

In [ ]:
class GPT(nn.Module):
    def __init__(self, block_size, vocab_size, d_model, n_heads, dff, n_layers, device):
        super().__init__()
        self.block_size= block_size
        self.token_embedding= nn.Embedding(vocab_size, d_model)
        self.position_embedding= nn.Embedding(block_size, d_model)

        self.blocks= nn.Sequential(*[
            GPTBlock(d_model, n_heads, dff) for _ in range(n_layers)
        ])

        self.norm= nn.LayerNorm(d_model)
        self.head= nn.Linear(d_model, vocab_size)
        self.device= device

    @torch.no_grad()
    def generate(self, idx, max_new_tokens,temperature, top_k= None):
        ''' 
        This function runs the inference on the input token 

        Args:
        idx: input token/list of tokens (T, )
        max_new_tokens: max allowed output token length (int)
        temperature: this allows to pick some uncharted token which have otherwise chosen the max likelihood token (float)
        top_k: this allows the model to sample to k tokens and pick the most likelihood of the them. 
        
        Returns: 
        This returns the output tensor from the model which is actually the tokens generated by the model applying autoregressive approach of token generation in a decoder-only style model. 

        idx: Tensor of shape (max_new_tokens, )
        
        '''

        for _ in range(max_new_tokens):
            idx_cond= idx[:, -self.block_size:]
            logits, _= self(idx_cond)
            logits= logits[:, -1, :]

            logits= logits/temperature
            if top_k is not None:
                values, indices= torch.topk(logits, top_k)
                logits_filtered= torch.full_like(logits, float("-inf"))
                logits_filtered.scatter_(1, indices, values)

                logits= logits_filtered


            probs= nn.functional.softmax(logits, dim=-1)
            next_token= torch.multinomial(probs, num_samples= 1)
            idx= torch.cat((idx, next_token), dim=1)
        return idx

    def forward(self, idx, targets= None):
        ''' 
        This is the gpt's part that takes the input tensor, convert to embedding, process it and outputs the logits, loss. 
        
        Args:
        idx: input tensor that is eventually converted to embedding for further processing -(B, T) 
        targets: target tensor which help evaluate the prediction from the model -(B, T) 
        

        Returns: 
        logits: Prediction from the model- (B*T, d_model)
        loss: Error generated against the true value - (float)
        '''


        B, T= idx.shape # idx.size: (B, T) --> Batch_size, Seq_length (Block_size here)

        tok_embed= self.token_embedding(idx)  ## tok_embed.size: (B, T, d_model)
        pos= torch.arange(T, device= self.device)

        pos_embed= self.position_embedding(pos) ## pos_embed.size: (T, d_model)

        x= tok_embed+ pos_embed # x.size: (B, T, d_model)
        x= self.blocks(x)
        x= self.norm(x)
        logits= self.head(x) # logits.size: (B, T, d_model)

        loss= None

        if targets is not None:
            B, T, C= logits.shape
            logits= logits.view(B*T, C)

            targets= targets.view(B*T)
            loss= nn.functional.cross_entropy(logits, targets)

        return logits, loss # logits.shape: (B*T, d_model) when training; (B, T, d_model) when inferencing -- 



In [11]:
class EarlyStop:
    def __init__(self, patience=10, min_delta=0):
        self.patience= patience
        self.counter=0
        self.best_loss= float('inf')

    def early_stop(self, loss):
        if loss<self.best_loss:
            self.best_loss= loss
            self.counter=0

        else:
            self.counter+=1

        if self.counter>=self.patience:
            return True

        return False

In [15]:
model= GPT(block_size, vocab_size, d_model, n_heads, dff, n_layers, device).to(device)
optimizer= torch.optim.AdamW(model.parameters(), lr= learning_rate)
early_stopping= EarlyStop()
#scheduler= torch.optim.lr_scheduler.StepLR(optimizer, step_size= 10, gamma= 0.1)

In [16]:
print(sum(p.numel() for p in model.parameters())/1e6, "M parameters")

2.719169 M parameters


In [17]:
for iter in range(max_iters):
    model.train()
    xb, yb= get_batch('train')
    logits, loss= model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (iter+1)%eval_intervals==0:
        #print(f"Step {iter+1}: loss {loss.item():.4f}")
        with torch.no_grad():
            total= 0
            for _ in range(50):
                xb, yb= get_batch("val")
                _, test_loss= model(xb, yb)
                total+= test_loss.item()

        print(f"Step {iter+1}: Train loss: {loss.item():4f} Test item: {total/50:4f}")

        if early_stopping.early_stop(test_loss):
            print("Invoked the Early Stopping. Stopping training !!")
            break

    #scheduler.step()

Step 500: Train loss: 2.153263 Test item: 2.201876
Step 1000: Train loss: 1.871743 Test item: 1.974726
Step 1500: Train loss: 1.731213 Test item: 1.860096
Step 2000: Train loss: 1.622805 Test item: 1.776544
Step 2500: Train loss: 1.503690 Test item: 1.705448
Step 3000: Train loss: 1.545848 Test item: 1.680899
Step 3500: Train loss: 1.464524 Test item: 1.641615
Step 4000: Train loss: 1.423771 Test item: 1.595417
Step 4500: Train loss: 1.323692 Test item: 1.599547
Step 5000: Train loss: 1.334565 Test item: 1.578442
Step 5500: Train loss: 1.311410 Test item: 1.572107
Step 6000: Train loss: 1.356877 Test item: 1.570434
Step 6500: Train loss: 1.265234 Test item: 1.568745
Step 7000: Train loss: 1.267387 Test item: 1.541557
Step 7500: Train loss: 1.304057 Test item: 1.556138
Step 8000: Train loss: 1.280428 Test item: 1.552643
Step 8500: Train loss: 1.205236 Test item: 1.534420
Step 9000: Train loss: 1.225429 Test item: 1.551716
Step 9500: Train loss: 1.197390 Test item: 1.557749
Step 10000: T

In [18]:
#context= torch.zeros((1, 1), dtype= torch.long, device= device)
context= torch.tensor([encode("ROMEO:\n")], dtype= torch.long, device= device)
print(decode(model.generate(context, 400, 0.8, 15)[0].tolist()))

ROMEO:
Bad no life, the dugger more than the way for meet what hate
Theirs our heads is confected with whence he
Thurd set this we through the world of twenty
And all the women with the lord what I have thee,
They butcher than was near in thy things may
Have but not die to the distressing rest.

KING RICHARD III:
Richard, and the traitor care of the king?

QUEEN MARGARET:
But a fearful time as such as fo


In [ ]:
class CausalSelfAttention(nn.Module):
    ''' 
    This implementation tries to achieve a custom form of Self attention from where we are able to take out our target kv to cache
    
    '''

    def __init__(self, d_model, n_heads):
        super().__init__()

        assert d_model % n_heads == 0
        self.nheads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x, past_k=None, past_v=None):

        ''' 
        This is the custom Attention Implementation so as to get the key,value required for building the kv-cache inference

        Args:
        x: (B, T, d_model) -- input tensor 
        past_k: (B, T, d_model) -- the stored or cached key embeddings used during inference
        past_v: (B, T, d_model) -- the stored or cache value embeddings used during inference

        Returns: 
        This returns the output embeddings from the model after application of attention and also returns the k and v useful for building the kv-cache

        out: (B, T, d_model) -- the weighted output from the transformer
        k: (B, T, d_model) or (B, T+1, d_model) -- the key to build the key cache
        v: (B, T, d_model) or (B, T+1, d_model) -- the value to build the value cache
        
        '''
        B, T, C = x.shape

        q = self.q_proj(x)  # (B, T, d_model)
        k = self.k_proj(x)  #  same
        v = self.v_proj(x)  #  same  

        q = q.view(B, T, self.nheads, self.head_dim).transpose(1, 2)    # (B, nheads, T, head_dim)
        k = k.view(B, T, self.nheads, self.head_dim).transpose(1, 2)    # same
        v = v.view(B, T, self.nheads, self.head_dim).transpose(1, 2)    # same

        # activated only during inference -- during training no use to implement kv-caching and thus to fasten the inference kv-cache used, thus the implementation
        if past_k is not None and past_v is not None: 
            k_full = torch.cat([past_k, k], dim=2)  # (B, nheads, T+1, head_dim)
            v_full = torch.cat([past_v, v], dim=2)  # same
        else:
            k_full = k  # (B, T, d_model)
            v_full = v

        attn = (q @ k_full.transpose(-2, -1)) / math.sqrt(self.head_dim)    # (B, nheads, T, T)

        # Apply causal mask only during training, applied to prevent the model from looking at future tokens and prevent unfair advantage to predict the next-token
        if past_k is None:
            mask = torch.triu(                      # (T, T)
                torch.ones(T, T, device=x.device),
                diagonal=1
            ).bool()    
            attn = attn.masked_fill(mask, float("-inf"))    # (B, nheads, T, T)

        attn = torch.softmax(attn, dim=-1)

        ## -- dot product taking place remember --
        out = attn @ v_full     # (B, nheads, T, head_dim)  
        out = out.transpose(1, 2).contiguous().view(B, T, C)    # (B, T, d_model) -- merging the nheads with the head_dim
        out = self.out_proj(out)    # (B, T, d_model)

        # return only incremental k, v
        return out, k, v


In [ ]:
class GPTBlock(nn.Module):
    ''' 
    This is the decoder block that takes up the input embeddings and outputs the embeddings after getting the output from the attention operation from the transformer
    
    '''
    def __init__(self, d_model, nheads, dff):
        super().__init__()

        self.norm1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, nheads)
        self.norm2 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, dff),
            nn.GELU(),
            nn.Linear(dff, d_model)
        )

    def forward(self, x, past_k=None, past_v=None):
        attn_out, new_k, new_v = self.attn(self.norm1(x), past_k, past_v) # already mentioned the dimensions of attn_out
        x = x + attn_out
        x = x + self.ff(self.norm2(x))
        return x, new_k, new_v


In [ ]:
class GPTModel(nn.Module):
    ''' 
    Flow of data:

    Input tensor >> (InputEmbedding + PositionalEncoding) >> CausalSelfAttention >> LayerNorm >> FeedForwardLayer >> Output Tensor
    
    '''

    def __init__(self, block_size, vocab_size, d_model, nheads, dff, nlayers):
        super().__init__()

        self.block_size = block_size

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)

        self.blocks = nn.ModuleList([
            GPTBlock(d_model, nheads, dff) for _ in range(nlayers)
        ])

        self.norm = nn.LayerNorm(d_model)
        self.ff = nn.Linear(d_model, vocab_size)

    def forward(self, idx, targets=None, past_kvs=None, pos_offset=0):
        B, T = idx.shape

        tok_embed = self.token_embedding(idx)

        # Sliding window position handling
        if T == 1 and past_kvs is not None:
            pos_val = past_kvs[0][0].size(2) % self.block_size
            pos = torch.tensor([pos_val], device=idx.device)
        else:
            pos = torch.arange(T, device=idx.device)

        pos_embed = self.position_embedding(pos)

        x = tok_embed + pos_embed

        new_kvs = []

        for i, block in enumerate(self.blocks):
            past_k = past_v = None
            if past_kvs is not None:
                past_k, past_v = past_kvs[i]

            x, new_k, new_v = block(x, past_k, past_v)
            new_kvs.append((new_k, new_v))

        x = self.norm(x)
        logits = self.ff(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(
                logits.view(B*T, C),
                targets.view(B*T)
            )

        return logits, loss, new_kvs

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        self.eval()

        logits, _, past_kvs = self(idx)

        for _ in range(max_new_tokens):

            current_token = idx[:, -1:]

            logits, _, new_kvs = self(
                current_token,
                past_kvs=past_kvs
            )

            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                values, indices = torch.topk(logits, top_k)
                logits_filtered = torch.full_like(logits, float("-inf"))
                logits_filtered.scatter_(1, indices, values)
                logits = logits_filtered

            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1)

            idx = torch.cat([idx, next_token], dim=1)

            # Update KV cache
            updated = []
            for (past_k, past_v), (new_k, new_v) in zip(past_kvs, new_kvs):

                past_k = torch.cat([past_k, new_k], dim=2)
                past_v = torch.cat([past_v, new_v], dim=2)

                # Trim to sliding window
                if past_k.size(2) > self.block_size:
                    past_k = past_k[:, :, -self.block_size:, :]
                    past_v = past_v[:, :, -self.block_size:, :]

                updated.append((past_k, past_v))

            past_kvs = updated

        return idx


In [29]:
model_kv_cache= GPTModel(block_size, vocab_size, d_model, n_heads, dff, n_layers)
print(f"Model created on device: {next(model_kv_cache.parameters()).device}")
model_kv_cache= model_kv_cache.to(device)
print(f"Model moved to: {device}")
optimizer= torch.optim.AdamW(model_kv_cache.parameters(), lr= learning_rate)
early_stopping= EarlyStop()

Model created on device: cpu
Model moved to: cuda


In [30]:
print(next(model_kv_cache.parameters()).device)

cuda:0


In [31]:
for iter in range(10000):
    model_kv_cache.train()
    xb, yb= get_batch('train')
    logits, loss, _= model_kv_cache(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (iter+1)%eval_intervals==0:
        #print(f"Step {iter+1}: loss {loss.item():.4f}")
        with torch.no_grad():
            total= 0
            for _ in range(50):
                xb, yb= get_batch("val")
                _, test_loss, _= model_kv_cache(xb, yb)
                total+= test_loss.item()

        print(f"Step {iter+1}: Train loss: {loss.item():4f} Test loss: {total/50:4f}")

        if early_stopping.early_stop(test_loss):
            print("Invoked the Early Stopping. Stopping training !!")
            break

    #scheduler.step()

Step 500: Train loss: 2.189507 Test loss: 2.162607
Step 1000: Train loss: 1.795580 Test loss: 1.899812
Step 1500: Train loss: 1.627905 Test loss: 1.795418
Step 2000: Train loss: 1.550329 Test loss: 1.690064
Step 2500: Train loss: 1.401158 Test loss: 1.643488
Step 3000: Train loss: 1.392899 Test loss: 1.611594
Step 3500: Train loss: 1.374758 Test loss: 1.578653
Step 4000: Train loss: 1.324090 Test loss: 1.559313
Step 4500: Train loss: 1.310133 Test loss: 1.556479
Step 5000: Train loss: 1.272047 Test loss: 1.556704
Step 5500: Train loss: 1.220141 Test loss: 1.539603
Step 6000: Train loss: 1.226791 Test loss: 1.532186
Step 6500: Train loss: 1.226760 Test loss: 1.542325
Step 7000: Train loss: 1.147667 Test loss: 1.544814
Step 7500: Train loss: 1.086280 Test loss: 1.553086
Step 8000: Train loss: 1.132186 Test loss: 1.555908
Step 8500: Train loss: 1.120301 Test loss: 1.590343
Step 9000: Train loss: 1.087907 Test loss: 1.594050
Invoked the Early Stopping. Stopping training !!


In [35]:
# Simple generation test — prints the decoded output and token count
context = torch.tensor([encode("ROMEO:\n")], dtype=torch.long, device=device)
#print("context:", decode(context[0].tolist()))
#print(context[0])
out = model_kv_cache.generate(context, 1000, 0.8, 15)
print(decode(out[0].tolist()))  
print("output token count:", out.size(1))

ROMEO:
DUKE VINCENTIO:
Good friend mam?

Provost:
Ay, mourdered me, and go well.

POLIXENES:
And in all the common-play, as wellllllllllllllllllllllllllllllllllllllllindeeeadeeelawhoreelareritherelliflindelikeeligorererithofarvelilelikeeeldeyififurtheervervint s teeivenimicesst sthegaivethenecoweaveesy, ye or isss s ires eewhet here pegss e brescert p therer mrenor tope prare fan, tureroa t de, imane tou a s omyouritotofan ounspeeta be are irtit heyithe forofurove sse t be p wonofe ale, tree tre helaithoofrourd hole d myod se ssprer s t gre be ou d tore thoresherenore ofof thamy be theriflinoure t shee iginou oure san t dim shandarean l moritomyou s hou wssprowitouiflaloussaiofove t be she ma soff e hereseanor fotorer ifar hosseligrire oue ore, s anofirabireaitheal da es ber tad t w tanondre are alof ougreachare dofofris bea ofedans a premedatirithanow llritonovist lig
Towa te tit:
teffe pea faithondovimy wre tousesor iraidsit het myougheanin ther a siraifomyove hee winondeea ssus tane